# CS604 Colab launcher

Runtime: GPU. Outputs go to Drive so they survive the session. Change `METHOD` and `TEAM` below before training / predict.

In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
DRIVE_OUT = "/content/drive/MyDrive/cs604-runs"
METHOD = "lora_r8"  # linear_probe | full_ft | lora_r8 | bitfit | adapter | adaptformer
TEAM = "xx"

## 1. Clone or pull the repo

In [ ]:
import os

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## Review settings (table)

Loads `configs/{METHOD}.yaml` (merged with `base.yaml`) so you can check every parameter before spending a GPU hour.

In [ ]:
import pandas as pd

from src.utils import apply_overrides, load_config

methods = ["full_ft", "linear_probe", "lora_r8", "bitfit", "adapter", "adaptformer"]
merged = {name: load_config(f"configs/{name}.yaml").to_dict() for name in methods}
keys = []
for d in merged.values():
    for k in d:
        if k not in keys:
            keys.append(k)

compare = pd.DataFrame(
    {name: [merged[name].get(k, "") for k in keys] for name in methods},
    index=keys,
)
compare.index.name = "parameter"
print("Resolved parameters per method (base.yaml already merged in):")
display(compare)

chosen = load_config(f"configs/{METHOD}.yaml")
apply_overrides(chosen, data_root=DATA_DIR, output_dir=DRIVE_OUT)
resolved = pd.DataFrame(list(chosen.to_dict().items()), columns=["parameter", "value"])
print(f"\nThis run (METHOD={METHOD!r}, plus Colab data_root / output_dir):")
display(resolved)

## 5. Train (checkpoints on Drive)

In [ ]:
!python -m src.train --config configs/{METHOD}.yaml --data_root {DATA_DIR} --output_dir {DRIVE_OUT}

## 6. Predict

Set `CKPT` to the `best.pt` (or `last.pt`) from the run folder printed above.

In [ ]:
CKPT = f"{DRIVE_OUT}/REPLACE_WITH_RUN_DIR/best.pt"

!python -m src.predict --config configs/{METHOD}.yaml --ckpt {CKPT} --split val --team {TEAM} --data_root {DATA_DIR} --out_dir {DRIVE_OUT}
!python -m src.predict --config configs/{METHOD}.yaml --ckpt {CKPT} --split test --team {TEAM} --data_root {DATA_DIR} --out_dir {DRIVE_OUT}